# Multimodal Embeddings with Gemini

This notebook explores `gemini-embedding-2-preview`, Google's first multimodal embedding model that supports text, images, video, audio, and PDFs.

## Features

1. **Multimodal Inputs**: Embed text, images, PDFs, video, and combinations in a single request
2. **Configurable Dimensions**: Control output embedding size via `output_dimensionality`
3. **Cosine Similarity**: Compare embeddings across modalities

## Reference
- [Gemini Embeddings Docs](https://ai.google.dev/gemini-api/docs/embeddings)

In [ ]:
# Setup
from google import genai
from google.genai import types
from pypdf import PdfReader, PdfWriter
import numpy as np
import io
import time

client = genai.Client()

MODEL = "gemini-embedding-2-preview"
MAX_PDF_PAGES = 6  # API limit for gemini-embedding-2-preview


def read_pdf_bytes(path, max_pages=MAX_PDF_PAGES):
    """Read a PDF, truncating to max_pages. Returns bytes."""
    reader = PdfReader(path)
    if len(reader.pages) <= max_pages:
        return path.read_bytes(), len(reader.pages)

    writer = PdfWriter()
    for page in reader.pages[:max_pages]:
        writer.add_page(page)
    buf = io.BytesIO()
    writer.write(buf)
    return buf.getvalue(), min(len(reader.pages), max_pages)


def embed(contents, dimensionality=None):
    """Embed content with latency tracking and optional config."""
    config = {}
    if dimensionality:
        config["output_dimensionality"] = dimensionality

    start = time.time()
    result = client.models.embed_content(
        model=MODEL,
        contents=contents,
        config=types.EmbedContentConfig(**config) if config else None,
    )
    latency = time.time() - start
    print(f"Latency: {latency:.2f}s | Embeddings: {len(result.embeddings)} | Dims: {len(result.embeddings[0].values)}")
    return result


def cosine_similarity(a, b):
    """Compute cosine similarity between two embedding vectors."""
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


print("Setup complete!")

## 1. Semantic Search over PDFs

Embed a collection of PDFs and find the most relevant one for a given text query — a core building block for RAG pipelines.

In [ ]:
from pathlib import Path

# Point this to a folder of PDFs you want to search over
PDF_DIR = Path("YOUR_PDF_FOLDER")
pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs in {PDF_DIR}")
for f in pdf_files:
    print(f"  - {f.name}")

In [ ]:
# Embed each PDF (truncated to first 6 pages)
pdf_embeddings = {}

for pdf_path in pdf_files:
    pdf_bytes, page_count = read_pdf_bytes(pdf_path)
    print(f"Embedding {pdf_path.name} ({page_count} pages)...")
    result = embed([
        types.Part.from_bytes(
            data=pdf_bytes,
            mime_type="application/pdf",
        )
    ])
    pdf_embeddings[pdf_path.name] = result.embeddings[0].values

print(f"\nEmbedded {len(pdf_embeddings)} documents.")

In [ ]:
# Search: rank PDFs by relevance to a text query
query = "multimodal retrieval augmented generation"

query_result = embed(query)
query_vec = query_result.embeddings[0].values

# Rank documents by cosine similarity
scores = {
    name: cosine_similarity(query_vec, doc_vec)
    for name, doc_vec in pdf_embeddings.items()
}
ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

print(f"Query: \"{query}\"\n")
print(f"{'Rank':<6}{'Score':<10}{'Document'}")
print("-" * 60)
for i, (name, score) in enumerate(ranked, 1):
    print(f"{i:<6}{score:<10.4f}{name}")

## 2. Video Embedding & Text-to-Video Search

Embed a video clip and match it against text descriptions. Useful for video cataloging, content moderation, or building a visual search engine.

**Limits**: max 128s duration per video.

In [ ]:
# Point this to a video file (MP4, MOV, etc.)
VIDEO_PATH = Path("YOUR_VIDEO_FILE.mp4")

with open(VIDEO_PATH, "rb") as f:
    video_bytes = f.read()

print(f"Video: {VIDEO_PATH.name} ({len(video_bytes) / 1024 / 1024:.1f} MB)")

video_result = embed([
    types.Part.from_bytes(data=video_bytes, mime_type="video/mp4")
])
video_vec = video_result.embeddings[0].values
print(f"First 5 values: {video_vec[:5]}")

In [ ]:
# Match the video against candidate text descriptions
candidates = [
    "A person giving a presentation about machine learning",
    "A cat playing with a ball of yarn",
    "A scenic mountain landscape with a river",
    "A cooking tutorial showing how to make pasta",
    "A software demo showing a web application",
    "A cat playing with the phone"
]

# Embed all candidates in one call
candidate_results = embed(candidates)

print(f"Video: {VIDEO_PATH.name}\n")
print(f"{'Score':<10}{'Description'}")
print("-" * 60)

scores = []
for desc, emb in zip(candidates, candidate_results.embeddings):
    score = cosine_similarity(video_vec, emb.values)
    scores.append((score, desc))

for score, desc in sorted(scores, reverse=True):
    print(f"{score:<10.4f}{desc}")

## 3. Cross-Modal Similarity Matrix

Compare embeddings across modalities (text, PDF, video) to see how well the shared embedding space aligns different content types.

In [ ]:
import pandas as pd

# Collect all embeddings into a labeled dict
all_embeddings = {}

# Text queries
text_queries = [
    "table detection in documents",
    "financial data extraction",
]
for q in text_queries:
    r = embed(q)
    all_embeddings[f"text: {q[:30]}"] = r.embeddings[0].values

# PDFs (reuse from section 1)
for name, vec in list(pdf_embeddings.items())[:3]:  # limit to first 3
    all_embeddings[f"pdf: {name[:30]}"] = vec

# Video (reuse from section 2)
all_embeddings[f"video: {VIDEO_PATH.name}"] = video_vec

# Build similarity matrix
labels = list(all_embeddings.keys())
vecs = list(all_embeddings.values())
n = len(labels)

matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        matrix[i][j] = cosine_similarity(vecs[i], vecs[j])

df = pd.DataFrame(matrix, index=labels, columns=labels).round(3)
print("Cross-modal similarity matrix:\n")
df

## 4. Effect of Output Dimensionality on Similarity

Test whether reducing embedding dimensions (for storage/speed) preserves ranking quality.

In [ ]:
# Compare ranking stability across different embedding sizes
query_text = "multimodal retrieval augmented generation"
dims_to_test = [256, 512, 768, 3072]  # 3072 is the full default

print(f"Query: \"{query_text}\"\n")

for dim in dims_to_test:
    # Embed query
    q = embed(query_text, dimensionality=dim)
    q_vec = q.embeddings[0].values

    # Embed all PDFs at this dimensionality
    doc_scores = {}
    for pdf_path in pdf_files:
        pdf_bytes, _ = read_pdf_bytes(pdf_path)
        r = embed(
            [
                types.Part.from_bytes(
                    data=pdf_bytes,
                    mime_type="application/pdf",
                )
            ],
            dimensionality=dim,
        )
        doc_scores[pdf_path.name] = cosine_similarity(q_vec, r.embeddings[0].values)

    ranked = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    print(f"\n--- dims={dim} ---")
    for i, (name, score) in enumerate(ranked, 1):
        print(f"  {i}. {score:.4f}  {name}")